# Phase 1: Analytical SQL — Production Database Queries from Python

## Section 0: Setup & Database Connection

### Why Analytical SQL in Python?
In modern enterprise data science and analytics engineering, datasets rarely exist as convenient, static `.csv` files stored on a local hard drive. Instead, operational platforms record transactions inside cloud-hosted relational databases such as **Supabase (PostgreSQL)**.

Rather than exporting massive tables to disk or overloading local machine memory by loading entire database dumps into Python, professional data teams use **Analytical SQL** to push compute (filtering, joining, windowing, and aggregating) directly to the database engine. Python then orchestrates the pipeline, ingesting only the refined, analytical slices needed for modeling, customer segmentation, or dashboard reporting.

### Core Architectural Components:
1. **SQLAlchemy Engine (`create_engine`)**:
   - The engine is the central connection factory and pool manager between Python and the database.
   - It manages a persistent **connection pool** (reusing established network connections rather than incurring high TLS/TCP handshake latency for every query), abstracts dialect differences between SQL flavors, and handles connection lifecycles.
2. **Environment Variables & `python-dotenv` (`load_dotenv`)**:
   - Connecting to a database requires sensitive credentials: username, password, host URI, port, and database name.
   - Hardcoding connection strings directly in notebooks exposes passwords to version control (e.g., GitHub), creating severe security vulnerabilities.
   - `python-dotenv` reads environment variables from a local `.env` file into process memory (`os.environ`), adhering strictly to **Twelve-Factor App** security principles.
3. **`pandas.read_sql()`**:
   - Bridges SQL and Pandas by executing a SQL query (or SQLAlchemy `text()` clause) against the database engine.
   - Streams the database cursor result, maps PostgreSQL data types (timestamps, numerics, text) into appropriate Pandas data types, and returns a clean, structured `pd.DataFrame` ready for downstream analysis.

> **Interview Insight**: In technical interviews, emphasize that executing aggregations and filtering in PostgreSQL server-side before loading data into pandas minimizes network bandwidth, optimizes query execution via database query planners/indexes, and prevents Python Out-Of-Memory (`OOM`) crashes when handling multi-gigabyte transactional tables.

In [ ]:
# Section 0: Setup, Imports, and Connection Verification
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, text
import pandas as pd

# Load environment variables from the root .env file
load_dotenv()

# Initialize the SQLAlchemy engine for Supabase PostgreSQL
engine = create_engine(os.getenv('SUPABASE_URL'))

# Run a lightweight health check query to verify database connectivity
test_query = text("SELECT 1 AS test")
df_test = pd.read_sql(test_query, engine)

print("Database connection successfully established!")
print("Diagnostic query result:")
print(df_test)

## Section 1: Multi-Table JOIN — The Customer Order Journey

### What is a Relational JOIN and Why Do We Need It?
Production transactional databases (OLTP) are structured in **Third Normal Form (3NF)** to eliminate data redundancy, enforce referential integrity, and optimize write throughput. For instance, customer demographic data is stored once in `customers`, order metadata in `orders`, item prices in `order_items`, and catalog attributes in `products`. 

To analyze the end-to-end **Customer Order Journey**—tracking where a customer lives, what order they placed, which specific product item they bought, its translated category, and the final payment collected—we must join normalized tables along Primary Key (PK) to Foreign Key (FK) relationships.

### INNER JOIN vs. LEFT JOIN:
- **`INNER JOIN`**: Returns rows only when there is an exact key match in *both* tables. If an order has no matching items or payments, it is excluded. Use `INNER JOIN` when strict referential integrity is guaranteed and you only care about fully fulfilled transactions.
- **`LEFT JOIN` (Outer Join)**: Preserves every single record from the left table, populating `NULL` values for columns from the right table if no match is found. Use `LEFT JOIN` when related data is optional or incomplete (e.g., product category translation might be missing for new items) to prevent unintentional record dropping.

### The 6-Table Customer Journey Schema:
This query joins across 6 core tables in the Olist e-commerce database:
1. `customers` (demographics: `customer_city`, `customer_state`)
2. `orders` (order headers: `order_id`, `order_status`, `order_purchase_timestamp`)
3. `order_items` (order line item details: `price`, `product_id`)
4. `products` (catalog lookup: `product_category_name` in Portuguese)
5. `category_translation` (bilingual mapping: `product_category_name_english`)
6. `payments` (revenue settlement: `payment_value`)

> **Interview Insight**: Watch out for **Cartesian products and fan-out errors** when joining multiple 1-to-many tables (e.g., an order with 3 items and 2 split payment tenders produces 3 × 2 = 6 rows in a naive join). In enterprise analytics, data engineers aggregate child tables in Common Table Expressions (CTEs) before joining.

In [ ]:
# Section 1: Multi-Table JOIN Query — The Customer Order Journey
query_journey = text("""
SELECT 
    c.customer_id,
    c.customer_city,
    c.customer_state,
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp::date AS order_date,
    ct.product_category_name_english AS category,
    oi.price,
    p.payment_value
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products pr ON oi.product_id = pr.product_id
LEFT JOIN category_translation ct ON pr.product_category_name = ct.product_category_name
JOIN payments p ON o.order_id = p.order_id
LIMIT 1000;
""")

# Execute the query and load into df_journey
df_journey = pd.read_sql(query_journey, engine)

# Inspect DataFrame dimensions and top records
print(f"df_journey shape: {df_journey.shape} (Rows, Columns)")
print("\nFirst 5 Records:")
print(df_journey.head(5))

## Section 2: Window Functions — Customer Lifetime Value & Order Rank

### What are Window Functions?
A **Window Function** performs calculations across a set of table rows that are related to the current row (termed the "window frame"). 

### The Key Distinction: `GROUP BY` vs. Window Functions
- **`GROUP BY` collapses rows**: It aggregates multiple rows into a single summary record per group. In doing so, row-level granularity, timestamps, and individual transaction IDs are eliminated.
- **Window functions preserve row-level granularity**: They calculate aggregates, running totals, or rankings over defined partitions while **keeping every original row intact** in the output.

### Anatomy of the Window Functions Used:
1. **`PARTITION BY customer_id`**:
   Divides the dataset into logical partitions per customer, resetting the calculation boundary for each customer.
2. **`SUM(payment_value) OVER (PARTITION BY customer_id)`**:
   Calculates the overall **Customer Lifetime Value (LTV)** across all purchases made by that customer, broadcasting that total across every order row for that customer.
3. **`SUM(payment_value) OVER (PARTITION BY customer_id ORDER BY order_purchase_timestamp::timestamp ROWS UNBOUNDED PRECEDING)`**:
   Calculates a **running cumulative spend** trajectory over time. The frame specification `ROWS UNBOUNDED PRECEDING` directs PostgreSQL to accumulate spend from the customer's very first order up through the current transaction row.
4. **`DENSE_RANK() OVER (PARTITION BY customer_id ORDER BY order_purchase_timestamp::timestamp)`**:
   Chronologically indexes each order for a customer (1st order, 2nd order, 3rd order).

> **Interview Insight — `ROW_NUMBER()` vs `RANK()` vs `DENSE_RANK()`**:
> - `ROW_NUMBER()`: Always generates unique sequential integers (1, 2, 3, 4); breaks ties arbitrarily.
> - `RANK()`: Assigns identical rank to ties, then skips subsequent numbers (e.g., 1, 2, 2, 4).
> - `DENSE_RANK()`: Assigns identical rank to ties without skipping numbers (e.g., 1, 2, 2, 3). Ideal for business-level purchase numbering.

In [ ]:
# Section 2: Window Functions Query — Customer Lifetime Value & Order Rank
query_window = text("""
WITH customer_orders AS (
    SELECT 
        o.customer_id,
        o.order_id,
        o.order_purchase_timestamp::timestamp AS order_purchase_timestamp,
        p.payment_value
    FROM orders o
    JOIN payments p ON o.order_id = p.order_id
)
SELECT 
    customer_id,
    order_id,
    order_purchase_timestamp,
    payment_value,
    SUM(payment_value) OVER (
        PARTITION BY customer_id
    ) AS lifetime_value,
    SUM(payment_value) OVER (
        PARTITION BY customer_id 
        ORDER BY order_purchase_timestamp::timestamp 
        ROWS UNBOUNDED PRECEDING
    ) AS running_spend,
    DENSE_RANK() OVER (
        PARTITION BY customer_id 
        ORDER BY order_purchase_timestamp::timestamp
    ) AS order_number
FROM customer_orders
ORDER BY customer_id, order_number
LIMIT 500;
""")

# Execute window query and load into df_window
df_window = pd.read_sql(query_window, engine)

print(f"df_window shape: {df_window.shape}")
print("\nFirst 10 Records with Lifetime Value, Running Spend, and Order Numbering:")
print(df_window.head(10))

## Section 3: CTE (Common Table Expression) — Seller Performance Dashboard

### What is a CTE (Common Table Expression)?
A **Common Table Expression (CTE)** is a temporary, named result set defined using the `WITH` clause that exists only during the execution scope of a single query.

### Why Use CTEs Instead of Subqueries?
1. **Readability & Linear Narrative**: Deeply nested subqueries force engineers to read SQL from the inside out. CTEs break complex pipelines into sequential, top-to-bottom named blocks.
2. **Modularity & Reusability**: A CTE can be referenced multiple times within downstream queries (e.g., self-joins or cross-comparisons against baseline averages).
3. **Ease of Debugging**: You can inspect each CTE independently in isolation before combining them into final joins.
4. **Analogy to Python**: Writing a CTE is conceptually identical to creating named intermediate variables in Python:
   ```python
   df_seller_rev = compute_seller_revenue()
   df_seller_revs = compute_seller_reviews()
   df_dashboard = df_seller_rev.merge(df_seller_revs, on='seller_id')
   ```

### Business Case: The Seller 360 Dashboard
Marketplace operations teams need to monitor merchant quality across two separate operational dimensions:
- **Sales Volume & Revenue**: Total distinct orders, gross merchandise value (GMV), and average order value (AOV).
- **Customer Satisfaction**: Total review count and average customer review score (1 to 5 stars).

Joining `order_items` directly with `reviews` without pre-aggregation creates a fan-out hazard (an order with multiple review updates or multiple line items would duplicate revenue figures). Creating two isolated CTEs—`seller_revenue` and `seller_reviews`—and joining their aggregated results guarantees mathematically accurate figures.

> **Interview Insight**: CTEs signal to interviewers that you design clean, maintainable, modular SQL pipelines that follow software engineering best practices rather than writing brittle, monolithic subqueries.

In [ ]:
# Section 3: CTE Query — Seller Performance Dashboard (Top 20 Sellers)
query_sellers = text("""
WITH seller_revenue AS (
    SELECT 
        seller_id,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(price) AS total_revenue,
        AVG(price) AS avg_order_value
    FROM order_items
    GROUP BY seller_id
),
seller_reviews AS (
    SELECT 
        oi.seller_id,
        AVG(r.review_score) AS avg_review_score,
        COUNT(*) AS total_reviews
    FROM order_items oi
    JOIN reviews r ON oi.order_id = r.order_id
    GROUP BY oi.seller_id
)
SELECT 
    sr.seller_id,
    sr.total_orders,
    ROUND(sr.total_revenue::numeric, 2) AS total_revenue,
    ROUND(sr.avg_order_value::numeric, 2) AS avg_order_value,
    ROUND(srev.avg_review_score::numeric, 2) AS avg_review_score,
    srev.total_reviews
FROM seller_revenue sr
JOIN seller_reviews srev ON sr.seller_id = srev.seller_id
ORDER BY sr.total_orders DESC
LIMIT 20;
""")

# Execute CTE query and load into df_sellers
df_sellers = pd.read_sql(query_sellers, engine)

print("Top 20 Sellers Performance Dashboard (Revenue vs Review Score):")
print(df_sellers)

## Section 4: Delivery Performance Analysis by State

### Date Arithmetic & Temporal Computations in PostgreSQL
In e-commerce operations, delivery speed and Service Level Agreement (SLA) reliability directly drive customer retention and marketplace ratings. PostgreSQL provides powerful temporal primitives:
1. **Timestamp Type Casting (`::timestamp`)**: Converts text timestamps into native date/time representations.
2. **Interval Arithmetic**: Subtracting two timestamps (`order_delivered_customer_date::timestamp - order_purchase_timestamp::timestamp`) produces a PostgreSQL `INTERVAL`.
3. **`EXTRACT(EPOCH FROM interval)`**: Converts any interval into total elapsed seconds.
4. **Seconds to Days Conversion**: Dividing total epoch seconds by `86400.0` (60 sec × 60 min × 24 hrs) computes fractional delivery days.
5. **Conditional Aggregation (`CASE WHEN` inside `SUM`)**:
   ```sql
   SUM(CASE WHEN order_delivered_customer_date::timestamp > order_estimated_delivery_date::timestamp THEN 1 ELSE 0 END)::float / COUNT(*) * 100
   ```
   Calculates the percentage of delivered orders that breached the estimated delivery date promised at checkout.

### Business Value: Identifying Supply Chain Bottlenecks
Brazil has continental dimensions and vast regional logistics disparities. States in the Southeast (São Paulo, Rio de Janeiro, Minas Gerais) benefit from dense highway networks and nearby fulfillment centers. Conversely, northern and northeastern states (Amazonas, Roraima, Amapá) face severe multi-modal shipping hurdles.

Analyzing delivery duration and SLA breach rates by customer state enables marketplace operations to:
- Adjust realistic delivery date estimates on product pages to manage buyer expectations.
- Identify underperforming regional carrier partners.
- Strategically position regional distribution hubs.

> **Interview Insight**: Conditional aggregation (`SUM(CASE WHEN...) / COUNT(*)`) is among the most frequently tested patterns in technical SQL screenings. Interviewers look for proper `NULL` filtering (`WHERE order_delivered_customer_date IS NOT NULL`) and safe floating-point division.

In [ ]:
# Section 4: Delivery Performance Query — State-by-State SLA & Delivery Duration
query_delivery = text("""
SELECT 
    c.customer_state,
    COUNT(*) AS total_orders,
    ROUND(AVG(EXTRACT(EPOCH FROM (o.order_delivered_customer_date::timestamp - o.order_purchase_timestamp::timestamp)) / 86400)::numeric, 2) AS avg_delivery_days,
    ROUND((SUM(CASE WHEN o.order_delivered_customer_date::timestamp > o.order_estimated_delivery_date::timestamp THEN 1 ELSE 0 END)::float / COUNT(*) * 100)::numeric, 2) AS late_delivery_pct
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY late_delivery_pct DESC;
""")

# Execute delivery analysis query and load into df_delivery
df_delivery = pd.read_sql(query_delivery, engine)

print("--- Top 10 States with Highest Late Delivery Rate (Logistics Bottlenecks) ---")
print(df_delivery.head(10))

print("\n--- Bottom 5 States with Lowest Late Delivery Rate (Top Logistics Performers) ---")
print(df_delivery.tail(5))

## Section 5: Payment Method Analysis & Basket Size Bucketing

### Core Analytical Concepts: `GROUP BY`, `HAVING`, and `CASE WHEN` Bucketing
1. **Multi-Column `GROUP BY`**:
   Groups transactions simultaneously by multiple dimensions (e.g., order ID and tender type) to calculate granular basket totals before downstream cohorting.
2. **`HAVING` vs. `WHERE`**:
   - `WHERE` filters individual input rows *prior* to grouping.
   - `HAVING` filters aggregated group metrics *after* the `GROUP BY` operation.
3. **Dynamic Cohort Bucketing with `CASE WHEN`**:
   Continuous variables (like order payment values) are often best analyzed by segmenting them into meaningful business tiers:
   - **Low**: `< R$ 100` (low-value impulse buys, accessories)
   - **Mid**: `R$ 100 - R$ 500` (typical retail purchases, apparel, home goods)
   - **High**: `R$ 500+` (high-ticket items: consumer electronics, appliances, furniture)

### Business Application: Payment Preferences Vary by Basket Size
Consumer payment behavior in Brazil exhibits strong bifurcations depending on ticket size:
- **Boleto Bancário & Debit**: Widely utilized for smaller, routine transactions by unbanked or cash-preferring consumers.
- **Credit Cards**: Heavily dominate high-value tiers because consumers rely on installment financing (`parcelas`) to distribute the cost of expensive goods across multiple monthly payments.

### Combining SQL Aggregation with Pandas Pivoting
We use SQL to execute the heavy aggregation and binning server-side, and then leverage `pandas.pivot_table()` to reshape the tidy output into an executive matrix format.

> **Interview Insight**: Knowing when to stop in SQL and transition to Python demonstrates practical data science acumen. Use SQL for set-based computation and heavy aggregation; use Pandas for cross-tabulation, presentation formatting, and matrix manipulation.

In [ ]:
# Section 5: Payment Method Analysis Query — Basket Value Bucketing
query_payment = text("""
WITH order_totals AS (
    SELECT 
        order_id,
        SUM(payment_value) AS total_value,
        payment_type
    FROM payments
    GROUP BY order_id, payment_type
),
bucketed AS (
    SELECT 
        payment_type,
        CASE 
            WHEN total_value < 100 THEN 'Low (<100)'
            WHEN total_value < 500 THEN 'Mid (100-500)'
            ELSE 'High (500+)'
        END AS value_tier,
        COUNT(*) AS order_count
    FROM order_totals
    GROUP BY payment_type, value_tier
)
SELECT 
    payment_type,
    value_tier,
    order_count
FROM bucketed
ORDER BY payment_type, value_tier;
""")

# Execute payment bucketing query
df_payment = pd.read_sql(query_payment, engine)

print("--- Payment Method by Value Tier (Raw SQL Output) ---")
print(df_payment)

# Reshape into an executive cross-tabulation matrix using Pandas pivot_table
pivot_payment = df_payment.pivot_table(
    index='payment_type',
    columns='value_tier',
    values='order_count',
    fill_value=0
)[['Low (<100)', 'Mid (100-500)', 'High (500+)']]

print("\n--- Payment Distribution Matrix (Pandas Pivot Table) ---")
print(pivot_payment)

## Section 6: Summary & Analytical SQL Interview Mastery

### Core Analytical SQL Patterns Mastered in this Notebook:
1. **Multi-Table Relational JOINs**:
   - Navigated normalized 3NF schemas across 6 tables (`customers`, `orders`, `order_items`, `products`, `category_translation`, `payments`).
   - Distinguished between `INNER JOIN` (strict referential match) and `LEFT JOIN` (preserving records with optional mappings).
2. **Window Functions**:
   - Computed Customer Lifetime Value (LTV) using `SUM() OVER (PARTITION BY ...)` without collapsing row granularity.
   - Tracked cumulative spend trajectories using `ROWS UNBOUNDED PRECEDING`.
   - Sequentially indexed orders per customer using `DENSE_RANK()`.
3. **Common Table Expressions (CTEs)**:
   - Replaced complex nested subqueries with clean, modular, top-to-bottom procedural SQL.
   - Prevented fan-out row multiplication when joining orthogonal operational metrics (seller revenue vs seller reviews).
4. **PostgreSQL Date & Temporal Arithmetic**:
   - Mastered timestamp casting, interval calculations, and epoch conversion via `EXTRACT(EPOCH FROM ...)`.
   - Computed SLA violation rates and regional fulfillment duration metrics.
5. **Conditional Aggregation & Dynamic Bucketing**:
   - Segmented continuous metrics into business cohorts using `CASE WHEN`.
   - Combined server-side SQL binning with Pandas matrix pivoting for executive presentation.

---

> ### 💡 The 80/20 Rule for Data Science & Analyst SQL Interviews
> **These 5 patterns cover ~80% of real-world SQL interview questions for data analyst and data scientist roles.**
>
> In technical interviews at top-tier tech companies, interviewers rarely test obscure SQL keywords. Instead, they test whether you understand:
> - **Data Granularity**: Knowing when a join will multiply rows and how to prevent it.
> - **Window Context**: Knowing how to compute customer-level metrics without losing order-level details.
> - **Business Logic in SQL**: Translating operational definitions (SLA breach, value tiers, lifetime value) into robust conditional statements (`CASE WHEN`).
> - **Code Quality**: Writing clean, readable CTE-driven queries that colleagues can maintain.

In [ ]:
# Section 6: Pipeline Execution Summary and Connection Cleanup
summary_data = [
    {"Pattern": "1. Multi-Table JOIN", "DataFrame": "df_journey", "Rows": len(df_journey), "Columns": len(df_journey.columns)},
    {"Pattern": "2. Window Functions", "DataFrame": "df_window", "Rows": len(df_window), "Columns": len(df_window.columns)},
    {"Pattern": "3. CTE Seller Dashboard", "DataFrame": "df_sellers", "Rows": len(df_sellers), "Columns": len(df_sellers.columns)},
    {"Pattern": "4. Delivery Performance", "DataFrame": "df_delivery", "Rows": len(df_delivery), "Columns": len(df_delivery.columns)},
    {"Pattern": "5. Payment Method Bucketing", "DataFrame": "df_payment", "Rows": len(df_payment), "Columns": len(df_payment.columns)}
]

df_summary = pd.DataFrame(summary_data)

print("=" * 75)
print("          ANALYTICAL SQL PIPELINE EXECUTION SUMMARY")
print("=" * 75)
print(df_summary.to_string(index=False))
print("=" * 75)

# Cleanly close and dispose of the SQLAlchemy connection pool
engine.dispose()
print("\nDatabase connection pool successfully disposed.")